# Negation detection: SQL-side vs Python classifier

Apples-to-apples comparison feeding ADR 0026. Two test queries (pulmonary embolism and lung cancer),
two filter approaches each:

1. **SQL-side**: `NOT REGEXP_LIKE` with `[^.;:]{0,40}` sentence-window scope for both negation and historical cues (the `scout-system-prompt.md` pattern).
2. **Python**: per-sentence classifier from the POC's `apply_negation`, restricted here to negation + historical cues (no uncertainty filter) so the comparison reflects what SQL actually does.

For each query we compute SQL ↔ Python agreement, a categorical breakdown of disagreement reasons, and surface disagreement rows with sentence-scoped highlighting for manual review.

Run on dev02 JupyterHub.

## Setup

In [ ]:
import os
import re
from typing import Any, Optional

import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option('display.max_colwidth', 500)
pd.set_option('display.width', 200)

# Trino connection. Uses the env vars Jupyter sets per user on dev02.
_engine = create_engine(
    f"trino://{os.environ['TRINO_USER']}@{os.environ['TRINO_HOST']}:{os.environ['TRINO_PORT']}/"
    f"{os.environ['TRINO_CATALOG']}/{os.environ['TRINO_SCHEMA']}?http_scheme={os.environ['TRINO_SCHEME']}"
)


def query(sql_str, params=None):
    """Run a SQL query against Scout's Trino. Returns a DataFrame.

    When `params` is empty, the SQL is passed through as a raw string so
    colons inside regex literals like `(?:...)` aren't mistaken for bind
    parameters by SQLAlchemy's `text()` parser.
    """
    if params:
        return pd.read_sql(text(sql_str), _engine, params=params)
    return pd.read_sql(sql_str, _engine)

## Inlined Python classifier

Copy of the relevant pieces of the POC's classifier, restricted to negation + historical cues
so the comparison matches the SQL filter scope (uncertainty is intentionally not checked here).

In [ ]:
NEGATION_PATTERNS = [
    r"\bno\s+(mri?\s+)?evidence\b",
    r"\bwithout\s+(mri?\s+)?evidence\b",
    r"\bnegative\s+for\b",
    r"\babsence\s+of\b",
    r"\bruled?\s+out\b",
    r"\brules?\s+out\b",
    r"\bexcluding\b",
    r"\bexcluded\b",
    r"\bno\s+\w+\s+(suggest|indication|sign)\b",
    r"\bnot\s+(seen|present|identified|visualized|noted|appreciated)\b",
    r"\b(?:does|did|do)\s+not\b",
    r"\bno\s+",
]

HISTORICAL_PATTERNS = [
    r"\bhistory\s+of\b",
    r"\bh/o\b",
    r"\bprior\b",
    r"\bprevious(?:ly)?\b",
    r"\bremote\b",
    r"\bs/p\b",
    r"\bstatus\s+post\b",
    r"\bpast\s+medical\s+history\b",
]

TERMINATION_PATTERNS = [
    r"\bbut\b",
    r"\bhowever\b",
    r"\bexcept\b",
    r"\balthough\b",
    r"\baside\s+from\b",
    r"\bother\s+than\b",
]

_NEG_RE = re.compile("|".join(NEGATION_PATTERNS), re.IGNORECASE)
_HIS_RE = re.compile("|".join(HISTORICAL_PATTERNS), re.IGNORECASE)
_TER_RE = re.compile("|".join(TERMINATION_PATTERNS), re.IGNORECASE)
_SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+(?=[A-Z])")

In [ ]:
def _split_sentences(text):
    parts = _SENTENCE_SPLIT_RE.split(text)
    return [p for p in parts if p.strip()]


def _excerpt(sentence, start, end, pad=60):
    s = max(0, start - pad)
    e = min(len(sentence), end + pad)
    return f"…{sentence[s:e].strip()}…"


def _classify_match(sentence, match_start, match_end):
    """Returns (label, excerpt). label ∈ {positive, negated, historical}.
    Uncertainty is intentionally NOT checked here so the comparison against
    SQL stays apples-to-apples."""
    pre = sentence[:match_start]
    post = sentence[match_end:]
    # Termination cue closes negation scope mid-sentence ("no X, but Y" → Y is positive)
    term_match = None
    for tm in _TER_RE.finditer(pre):
        term_match = tm
    pre_active = pre[term_match.end():] if term_match else pre
    if _NEG_RE.search(pre_active) or _NEG_RE.search(post):
        return "negated", _excerpt(sentence, match_start, match_end)
    if _HIS_RE.search(pre):
        return "historical", _excerpt(sentence, match_start, match_end)
    return "positive", _excerpt(sentence, match_start, match_end)


_PREFIX = {
    "negated": "[NEGATIVE]",
    "historical": "[HISTORICAL]",
}


def python_classify(text, term_pattern):
    """Returns (evidence, reason). evidence ∈ {'positive', 'flagged'}.

    Strict semantic matching the SQL NOT REGEXP_LIKE shape: row is flagged if
    ANY mention is negated or historical, even when positive mentions also exist.
    Positive only when NO mention is disqualified."""
    if not text:
        return "positive", "no text — classifier abstained"
    has_positive = False
    for sent in _split_sentences(text):
        for m in term_pattern.finditer(sent):
            label, excerpt = _classify_match(sent, m.start(), m.end())
            if label == "positive":
                has_positive = True
            else:
                return "flagged", f"{_PREFIX[label]} {excerpt}"
    if has_positive:
        return "positive", "positive mention found, no disqualifying cues"
    return "positive", "term not found in scanned text — classifier abstained"

## Comparison helpers

In [ ]:
from IPython.display import display, HTML


def classify_dataframe(df, term_pattern_str):
    """Apply Python classifier to a DataFrame. Returns (evidence_series, reason_series)."""
    pat = re.compile(term_pattern_str, re.IGNORECASE | re.DOTALL)
    evidence = []
    reason = []
    for _, row in df.iterrows():
        imp = row.get("report_section_impression") or ""
        fnd = row.get("report_section_findings") or ""
        text = imp + "\n" + fnd
        ev, rs = python_classify(text, pat)
        evidence.append(ev)
        reason.append(rs)
    return pd.Series(evidence, index=df.index), pd.Series(reason, index=df.index)


# ---------------------------------------------------------------------------
# HTML highlight display
# ---------------------------------------------------------------------------

def _html_escape(s):
    return (s.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
             .replace('"', '&quot;'))


def _badge(label, ok):
    color = '#2e7d32' if ok else '#c62828'
    text = 'KEEP' if ok else 'FLAG'
    return (f'<span style="background:{color};color:white;padding:1px 6px;'
            f'border-radius:3px;font-size:0.8em;font-family:monospace;'
            f'margin-right:4px;">{label}:{text}</span>')


_HIGHLIGHT_STYLES = {
    'term': 'background:#fff59d;font-weight:bold;padding:0 2px;border-radius:2px;',
    'neg':  'background:#ffcdd2;color:#b71c1c;font-weight:bold;padding:0 2px;border-radius:2px;',
    'his':  'background:#d7ccc8;color:#4e342e;font-style:italic;padding:0 2px;border-radius:2px;',
}


# Broader highlight patterns: match the bare cues both engines might consider,
# not just the Python classifier's strict phrases. This is for VISUAL highlight
# only; the classifier still uses _NEG_RE / _HIS_RE for the actual filter logic.
# Word-boundary lookbehind/lookahead prevents word-fragment matches ("no" in "nonobstructive").
HIGHLIGHT_NEG_RE = re.compile(
    r'(?<![a-zA-Z])(?:no|without|negative for|absence of|ruled? out|rules? out|'
    r'excludes?|excluded|excluding|denies?|denied|not (?:seen|present|identified|'
    r'visualized|noted|appreciated)|(?:does|did|do) not)(?![a-zA-Z])',
    re.IGNORECASE
)
HIGHLIGHT_HIS_RE = re.compile(
    r'(?<![a-zA-Z])(?:history of|h/o|prior|previous(?:ly)?|remote|status post|'
    r's/p|past medical history)(?![a-zA-Z])',
    re.IGNORECASE
)


def highlight_sentence(sentence, term_pattern):
    """Return HTML with the search term + negation/history cues colored.
    Uses the broader HIGHLIGHT_* patterns so cues either engine might match
    are visible, regardless of which engine flagged the row."""
    spans = []
    for m in term_pattern.finditer(sentence):
        spans.append((m.start(), m.end(), 'term'))
    for m in HIGHLIGHT_NEG_RE.finditer(sentence):
        spans.append((m.start(), m.end(), 'neg'))
    for m in HIGHLIGHT_HIS_RE.finditer(sentence):
        spans.append((m.start(), m.end(), 'his'))
    spans.sort(key=lambda x: (x[0], -(x[1] - x[0])))
    cleaned = []
    last_end = 0
    for s, e, c in spans:
        if s < last_end:
            continue
        cleaned.append((s, e, c))
        last_end = e
    out = []
    pos = 0
    for s, e, c in cleaned:
        if s > pos:
            out.append(_html_escape(sentence[pos:s]))
        out.append(f'<span style="{_HIGHLIGHT_STYLES[c]}">{_html_escape(sentence[s:e])}</span>')
        pos = e
    if pos < len(sentence):
        out.append(_html_escape(sentence[pos:]))
    return ''.join(out)


def show_sample_html(df, mask, label, term_pattern_str, n=15):
    """Render a sample of rows with sentence-scoped highlighting in Jupyter.
    Works for any boolean mask: disagreement, agreement (both kept, both excluded), etc."""
    pat = re.compile(term_pattern_str, re.IGNORECASE | re.DOTALL)
    total = int(mask.sum())
    rows = df[mask].head(n)
    parts = [
        f'<div style="font-family:system-ui;margin:0.5em 0;">'
        f'<h4 style="margin:0.5em 0;">{_html_escape(label)} '
        f'<span style="font-weight:normal;color:#666;">({min(n, total)} of {total} shown)</span></h4>'
    ]
    parts.append(
        '<div style="font-size:0.85em;color:#555;margin-bottom:0.4em;">'
        f'Legend: <span style="{_HIGHLIGHT_STYLES["term"]}">term</span> '
        f'<span style="{_HIGHLIGHT_STYLES["neg"]}">negation cue</span> '
        f'<span style="{_HIGHLIGHT_STYLES["his"]}">historical cue</span>'
        '</div>'
    )
    if rows.empty:
        parts.append('<div style="color:#999;">(none)</div></div>')
        display(HTML(''.join(parts)))
        return
    for _, r in rows.iterrows():
        imp = (r.get('report_section_impression') or '').strip()
        fnd = (r.get('report_section_findings') or '').strip()
        text = imp + '\n' + fnd
        sentences = [s for s in _split_sentences(text) if pat.search(s)]
        if not sentences:
            sentences = ['(term not found in scanned sentences — abstained)']
        meta = (f"mci={_html_escape(str(r['message_control_id']))} "
                f"acc={_html_escape(str(r.get('accession_number')))} "
                f"year={_html_escape(str(r.get('year')))}")
        badges = (
            _badge('SQL', bool(r.get('sql_kept_flag')))
            + _badge('Python', bool(r.get('py_kept')))
        )
        sentence_html = '<br>'.join(highlight_sentence(s, pat) for s in sentences)
        parts.append(
            '<div style="border-left:3px solid #90caf9;padding:0.4em 0.8em;'
            'margin:0.4em 0;background:#fafafa;border-radius:3px;">'
            f'<div style="font-family:monospace;font-size:0.78em;color:#666;margin-bottom:0.2em;">{meta}</div>'
            f'<div style="margin-bottom:0.3em;">{badges}</div>'
            f'<div style="line-height:1.4;">{sentence_html}</div>'
            '</div>'
        )
    parts.append('</div>')
    display(HTML(''.join(parts)))


# Back-compat alias so existing cells keep working
show_disagreement_html = show_sample_html


# ---------------------------------------------------------------------------
# Disagreement category breakdown
# ---------------------------------------------------------------------------

def disagreement_breakdown(df, query_label, neg_pat, hist_pat):
    """Categorize WHY rows disagreed between SQL and Python."""
    sql_kept_py_dropped = df[(df['sql_kept_flag']) & (~df['py_kept'])]
    sql_dropped_py_kept = df[(~df['sql_kept_flag']) & (df['py_kept'])]
    parts = [f'<h4 style="margin:0.5em 0;">{_html_escape(query_label)} — disagreement breakdown</h4>']

    # SQL kept, Python excluded — Python caught a cue SQL's regex window missed
    reasons = sql_kept_py_dropped['py_reason'].fillna('')
    neg_n = int(reasons.str.startswith('[NEGATIVE]').sum())
    his_n = int(reasons.str.startswith('[HISTORICAL]').sum())
    other_n = int(len(sql_kept_py_dropped) - neg_n - his_n)
    parts.append(
        f'<div style="margin:0.4em 0;font-size:0.9em;"><b>SQL kept, Python excluded</b> '
        f'({len(sql_kept_py_dropped)} rows)<br>'
        f'&nbsp;&nbsp;Python flagged on <b>negation</b>: {neg_n} '
        '<span style="color:#666;">(cue outside SQL\'s 40-char window, or cross-sentence)</span><br>'
        f'&nbsp;&nbsp;Python flagged on <b>historical</b>: {his_n} '
        '<span style="color:#666;">(same — Python\'s sentence scope vs SQL\'s 40-char window)</span><br>'
        f'&nbsp;&nbsp;Other / abstained: {other_n}'
        '</div>'
    )

    # SQL excluded, Python kept — which SQL pattern fired?
    text_series = (sql_dropped_py_kept['report_section_impression'].fillna('')
                   + '\n' + sql_dropped_py_kept['report_section_findings'].fillna(''))
    neg_fired = text_series.str.contains(neg_pat, regex=True, na=False)
    hist_fired = text_series.str.contains(hist_pat, regex=True, na=False)
    only_neg = int((neg_fired & ~hist_fired).sum())
    only_hist = int((~neg_fired & hist_fired).sum())
    both = int((neg_fired & hist_fired).sum())
    neither = int((~neg_fired & ~hist_fired).sum())
    parts.append(
        f'<div style="margin:0.4em 0;font-size:0.9em;"><b>SQL excluded, Python kept</b> '
        f'({len(sql_dropped_py_kept)} rows)<br>'
        f'&nbsp;&nbsp;SQL <b>negation</b> pattern fired only: {only_neg}<br>'
        f'&nbsp;&nbsp;SQL <b>history</b> pattern fired only: {only_hist}<br>'
        f'&nbsp;&nbsp;SQL <b>both</b> negation + history fired: {both}<br>'
        f'&nbsp;&nbsp;Neither (anomaly): {neither}'
        '</div>'
    )
    display(HTML(''.join(parts)))

## Test 1: Pulmonary embolism

High-volume negation test. Most chest CTs report on the absence of PE; this stresses the negation logic at scale.

In [ ]:
# Patterns. PE alone is too short for regex without word boundaries (Trino doesn't
# reliably support \b), so we match only the unambiguous full phrase.
#
# Pattern improvements vs the prior iteration:
# - `(?<![a-zA-Z])` fixed-width lookbehind before the cue alternation prevents
#   word-fragment matches (e.g., "no" inside "nonobstructive", "noted", "now").
#   Trino's Joni regex supports fixed-width lookbehind.
PE_TERM = r'(?i)pulmonary embolism'
PE_NEG = (
    r'(?is)(?<![a-zA-Z])(?:no|without|negative for|absence of|ruled? out|excludes?|denies?)'
    r'[^.;:]{0,40}pulmonary embolism'
)
PE_HIST = (
    r'(?is)(?<![a-zA-Z])(?:history of|prior|previous(?:ly)?|h/o|remote)'
    r'[^.;:]{0,40}pulmonary embolism'
)

# Single query: candidates + SQL filter outcome as a column. ORDER BY makes the
# LIMIT deterministic so both filters operate on the same exact row set.
pe_sql = f"""
WITH scored AS (
  SELECT message_control_id,
         primary_report_identifier,
         accession_number, year, modality, service_name,
         report_section_impression,
         report_section_findings,
         diagnoses,
         NOT (
           REGEXP_LIKE(COALESCE(report_section_impression, ''), '{PE_NEG}')
           OR REGEXP_LIKE(COALESCE(report_section_findings, ''), '{PE_NEG}')
           OR REGEXP_LIKE(COALESCE(report_section_impression, ''), '{PE_HIST}')
           OR REGEXP_LIKE(COALESCE(report_section_findings, ''), '{PE_HIST}')
         ) AS sql_kept_flag
  FROM delta.default.reports_latest_epic_view
  WHERE year = 2024
    AND REGEXP_LIKE(COALESCE(service_name, ''), '(?i)(chest|thorax|lung|pe ?ct|cta)')
    AND (
      REGEXP_LIKE(COALESCE(report_section_impression, ''), '{PE_TERM}')
      OR REGEXP_LIKE(COALESCE(report_section_findings, ''), '{PE_TERM}')
    )
)
SELECT * FROM scored
ORDER BY message_control_id
LIMIT 20000
"""

pe_df = query(pe_sql)
print(f"PE candidates fetched: {len(pe_df)}")
print(f"  sql_kept_flag=True:  {int(pe_df['sql_kept_flag'].sum())}")
print(f"  sql_kept_flag=False: {int((~pe_df['sql_kept_flag']).sum())}")

In [ ]:
# Apply Python classifier on the same rows
pe_df['py_evidence'], pe_df['py_reason'] = classify_dataframe(pe_df, PE_TERM)
pe_df['py_kept'] = pe_df['py_evidence'] == 'positive'

n = len(pe_df)
print(f"\n=== PE: {n} candidates ===")
print(f"  SQL kept:    {int(pe_df['sql_kept_flag'].sum()):6d}  ({100*pe_df['sql_kept_flag'].mean():.1f}%)")
print(f"  Python kept: {int(pe_df['py_kept'].sum()):6d}  ({100*pe_df['py_kept'].mean():.1f}%)")

agree = (pe_df['sql_kept_flag'] == pe_df['py_kept']).sum()
print(f"\n  SQL ↔ Python agreement: {agree}/{n} ({100*agree/n:.1f}%)")

In [ ]:
# Categorical breakdown of WHY SQL and Python disagree.
disagreement_breakdown(pe_df, 'PE', PE_NEG, PE_HIST)

In [ ]:
# Rows where SQL excluded but Python kept.
# Typically rows where SQL's regex window fired more broadly than Python's per-sentence scope.
mask = (~pe_df['sql_kept_flag']) & (pe_df['py_kept'])
show_disagreement_html(pe_df, mask, 'PE: SQL excluded, Python kept', PE_TERM, n=15)

In [ ]:
# Rows where SQL kept but Python excluded.
# Typically Python catches a negation/history cue SQL's [^.;:]{0,40} window missed.
mask = (pe_df['sql_kept_flag']) & (~pe_df['py_kept'])
show_disagreement_html(pe_df, mask, 'PE: SQL kept, Python excluded', PE_TERM, n=15)

In [ ]:
# Rows where both SQL and Python KEPT (agreement on positive). Sanity check that the
# things both engines agree to include look like real positive PE findings.
mask = (pe_df['sql_kept_flag']) & (pe_df['py_kept'])
show_sample_html(pe_df, mask, 'PE: Both kept (agreement on positive)', PE_TERM, n=10)

In [ ]:
# Rows where both SQL and Python EXCLUDED (agreement on negative). Sanity check that
# the things both engines agree to drop really do look like negated/historical mentions.
mask = (~pe_df['sql_kept_flag']) & (~pe_df['py_kept'])
show_sample_html(pe_df, mask, 'PE: Both excluded (agreement on negative)', PE_TERM, n=10)

## Test 2: Lung cancer

Multi-term compound query (the Epic example). Tests Boolean composition together with
negation and history exclusion. Synonym alternation across cancer/carcinoma/malignancy
and across adenocarcinoma/SCC/SCLC/small-cell/squamous cell.

In [ ]:
# Patterns. Three positive groups (lung/cancer/subtype) — must have at least one from each.
LC_LUNG = r'(?i)(?:pulmonary|lung)'
LC_CANCER = r'(?i)(?:cancer|carcinoma|malignan(?:cy|t))'
LC_SUBTYPE = r'(?i)(?:adenocarcinoma|squamous cell|small cell|SCC|SCLC)'

# Python classifier scans for any cancer-flavored token (not the broader lung/pulmonary terms).
LC_TERM = r'(?i)(?:carcinoma|adenocarcinoma|malignan(?:cy|t)|small cell|squamous cell|SCLC|SCC)'

# Pattern improvements vs the prior iteration:
# - `(?<![a-zA-Z])` lookbehind on the cue alternation prevents word-fragment matches.
# - LC_HIST anchor is narrowed: the prior version anchored on broad terms
#   (lung|pulmonary|carcinoma|cancer|adenocarcinoma|malignan), which caught phrases
#   like "history of breast cancer in patient with current lung carcinoma" as historical.
#   The new anchor only triggers on the specific cancer subtypes the query targets,
#   so "history of" + unrelated cancer mention doesn't fire.
LC_NEG = (
    r'(?is)(?<![a-zA-Z])(?:no|without|negative for|absence of|ruled? out|excludes?|denies?)'
    r'[^.;:]{0,40}(?:adenocarcinoma|squamous cell|small cell|SCC|SCLC|non-small cell|NSCLC)'
)
LC_HIST = (
    r'(?is)(?<![a-zA-Z])(?:history of|prior|previous(?:ly)?|h/o|remote)'
    r'[^.;:]{0,40}(?:adenocarcinoma|squamous cell|small cell|SCC|SCLC|non-small cell|NSCLC)'
)

lc_sql = f"""
WITH scored AS (
  SELECT message_control_id,
         primary_report_identifier,
         accession_number, year, modality, service_name,
         report_section_impression,
         report_section_findings,
         diagnoses,
         NOT (
           REGEXP_LIKE(COALESCE(report_section_impression, ''), '{LC_NEG}')
           OR REGEXP_LIKE(COALESCE(report_section_findings, ''), '{LC_NEG}')
           OR REGEXP_LIKE(COALESCE(report_section_impression, ''), '{LC_HIST}')
           OR REGEXP_LIKE(COALESCE(report_section_findings, ''), '{LC_HIST}')
         ) AS sql_kept_flag
  FROM delta.default.reports_latest_epic_view
  WHERE year = 2024
    AND REGEXP_LIKE(COALESCE(service_name, ''), '(?i)(chest|thorax|lung)')
    AND (
      REGEXP_LIKE(COALESCE(report_section_impression, ''), '{LC_LUNG}')
      OR REGEXP_LIKE(COALESCE(report_section_findings, ''), '{LC_LUNG}')
    )
    AND (
      REGEXP_LIKE(COALESCE(report_section_impression, ''), '{LC_CANCER}')
      OR REGEXP_LIKE(COALESCE(report_section_findings, ''), '{LC_CANCER}')
    )
    AND (
      REGEXP_LIKE(COALESCE(report_section_impression, ''), '{LC_SUBTYPE}')
      OR REGEXP_LIKE(COALESCE(report_section_findings, ''), '{LC_SUBTYPE}')
    )
)
SELECT * FROM scored
ORDER BY message_control_id
LIMIT 20000
"""

lc_df = query(lc_sql)
print(f"Lung cancer candidates fetched: {len(lc_df)}")
print(f"  sql_kept_flag=True:  {int(lc_df['sql_kept_flag'].sum())}")
print(f"  sql_kept_flag=False: {int((~lc_df['sql_kept_flag']).sum())}")

In [ ]:
lc_df['py_evidence'], lc_df['py_reason'] = classify_dataframe(lc_df, LC_TERM)
lc_df['py_kept'] = lc_df['py_evidence'] == 'positive'

n = len(lc_df)
print(f"\n=== Lung cancer: {n} candidates ===")
print(f"  SQL kept:    {int(lc_df['sql_kept_flag'].sum()):6d}  ({100*lc_df['sql_kept_flag'].mean():.1f}%)")
print(f"  Python kept: {int(lc_df['py_kept'].sum()):6d}  ({100*lc_df['py_kept'].mean():.1f}%)")

agree = (lc_df['sql_kept_flag'] == lc_df['py_kept']).sum()
print(f"\n  SQL ↔ Python agreement: {agree}/{n} ({100*agree/n:.1f}%)")

In [ ]:
disagreement_breakdown(lc_df, 'Lung cancer', LC_NEG, LC_HIST)

In [ ]:
mask = (lc_df['sql_kept_flag']) & (lc_df['py_kept'])
show_sample_html(lc_df, mask, 'Lung cancer: Both kept (agreement on positive)', LC_TERM, n=10)

In [ ]:
mask = (~lc_df['sql_kept_flag']) & (~lc_df['py_kept'])
show_sample_html(lc_df, mask, 'Lung cancer: Both excluded (agreement on negative)', LC_TERM, n=10)

In [ ]:
mask = (~lc_df['sql_kept_flag']) & (lc_df['py_kept'])
show_disagreement_html(lc_df, mask, 'Lung cancer: SQL excluded, Python kept', LC_TERM, n=15)

In [ ]:
mask = (lc_df['sql_kept_flag']) & (~lc_df['py_kept'])
show_disagreement_html(lc_df, mask, 'Lung cancer: SQL kept, Python excluded', LC_TERM, n=15)

## Summary

Two filtering approaches compared apples-to-apples on negation + historical detection only.
Uncertainty cues are excluded from the Python classifier here so the comparison reflects
what SQL actually does (`NOT REGEXP_LIKE` for negation and history; no uncertainty filter).

What to read from each test:

- **SQL ↔ Python agreement**: how often the two engines reach the same verdict on the same row.
  High agreement (>90%) means SQL's regex window and Python's per-sentence scope catch the same cases.
  Low agreement points to either a regex-quality issue or a structural difference in pattern scope.
- **Disagreement breakdown**: when they disagree, *why*. If "SQL excluded, Python kept" is dominated
  by the SQL history pattern firing, that pattern is broader than Python's term-anchored scope and
  may be over-aggressive for some research intents.
- **Per-row HTML samples**: show the matched term + cues in context so manual review can judge
  whether each disagreement is SQL being right, Python being right, or genuine ambiguity that
  the researcher would want to control per query.

Decision the data supports for ADR 0026:

- If agreement is high on the simple-phrase case (PE), the SQL approach is sufficient for that
  pattern of clinical query and the Python classifier earns no additional precision.
- If the multi-term case (lung cancer) shows lower agreement, the disagreement breakdown should
  attribute it to specific SQL pattern shapes (e.g., history-anchor list too broad). The fix
  lives in the system prompt's regex patterns, not in the engine choice.